In [1]:
import pandas as pd
import scipy.stats as st
import math as m
!pip install statsmodels
import statsmodels.stats.api as sm
import numpy as np

In [2]:
# Load the data
df = pd.read_csv("pizza_restaurant.csv")
df.head()

,Product Name,Crust,Toppings,Price,Delivery Time,# pizzas the customer ordered before
0,Pepperoni,Cheese_and_Garlic,4,17,26.3,4
1,Hawaiian,Cheese_and_Chili,4,17,27.8,4
2,Calzone,Cheese_and_Garlic,3,20,31.5,7
3,Margherita,Cheese,4,23,20.8,7
4,Calzone,Cheese_and_Garlic,4,19,27.7,8


In [3]:
#Summary Statistics
df.describe()

,Toppings,Price,Delivery Time,# pizzas the customer ordered before
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,3.965000,19.342000,25.061100,5.659000
std,1.021185,3.345479,2.490397,2.459831
min,1.000000,12.000000,17.800000,0.000000
25%,3.000000,17.000000,23.300000,4.000000
50%,4.000000,19.000000,25.100000,5.000000
75%,5.000000,21.000000,26.700000,7.000000
max,7.000000,33.000000,32.400000,15.000000


## Standard Error of Sample Mean

In [4]:
# With the formula: SD / sqrt(n) -> Price
print(df.Price.std()/ m.sqrt(df.Price.count()))
print(st.sem(df.Price))

0.105793327900337
0.105793327900337


## Standardization and z-score

In [5]:
# Us using the formula for Delivery Time
df['delivery_time_standardized'] = (df['Delivery Time'] - df['Delivery Time'].mean())/ df['Delivery Time'].std()


In [6]:
# Using Sklearn
from sklearn import preprocessing
df['delivery_time_standardized2'] = preprocessing.scale(df['Delivery Time'])

In [7]:
# Look at the data
df.describe()

,Toppings,Price,Delivery Time,# pizzas the customer ordered before,delivery_time_standardized,delivery_time_standardized2
count,1000.000000,1000.000000,1000.000000,1000.000000,1.000000e+03,1.000000e+03
mean,3.965000,19.342000,25.061100,5.659000,1.385558e-16,1.421085e-16
std,1.021185,3.345479,2.490397,2.459831,1.000000e+00,1.000500e+00
min,1.000000,12.000000,17.800000,0.000000,-2.915639e+00,-2.917098e+00
25%,3.000000,17.000000,23.300000,4.000000,-7.071562e-01,-7.075101e-01
50%,4.000000,19.000000,25.100000,5.000000,1.562000e-02,1.562781e-02
75%,5.000000,21.000000,26.700000,7.000000,6.580878e-01,6.584171e-01
max,7.000000,33.000000,32.400000,15.000000,2.946879e+00,2.948354e+00


In [8]:
# Value vary (very) slightly duo to imprecision of the function.
# The differences arre irrelevant


## Confidence Level

In [9]:
cl = [0.005, 0.025, 0.05, 0.95, 0.975, 0.995]
for alpha in cl:
    print(f'The normal distribution value for {alpha} is {round(st.norm.ppf(alpha),2)}')

The normal distribution value for 0.005 is -2.58
The normal distribution value for 0.025 is -1.96
The normal distribution value for 0.05 is -1.64
The normal distribution value for 0.95 is 1.64
The normal distribution value for 0.975 is 1.96
The normal distribution value for 0.995 is 2.58


## Confidence Intervals for Large Samples

In [16]:
# Confidence Interval for the Price Mean: mean +- z * SE
print(f"The mean is  {sample.Price.mean()}")
print(st.norm.interval(confidence = 0.95,
                loc = df.Price.mean(),
                scale = st.sem(df.Price)))

The mean is  19.1
(np.float64(19.134648887510703), np.float64(19.549351112489294))


In [11]:
# Using chatGPT, create a function that for each numerical variable that 
# has a sample size bigger than 30, computes confidence interval for the mean.

def calculate_confidence_intervals(df, confidence = 0.95):
    intervals = {}

    for column in df.select_dtypes(include=[np.number]).columns:
        data = df[column].dropna()
        if len(data) > 30:
            mean = np.mean(data)
            std_dev = np.std(data, ddof=1)
            interval = st.norm.interval(confidence, loc=mean, scale=std_dev/np.sqrt(len(data)))
            intervals[column] = interval

    return intervals

print(calculate_confidence_intervals(df))

{'Toppings': (np.float64(3.9017074909279676), np.float64(4.028292509072032)), 'Price': (np.float64(19.134648887510703), np.float64(19.549351112489294)), 'Delivery Time': (np.float64(24.9067464105456), np.float64(25.2154535894544)), '# pizzas the customer ordered before': (np.float64(5.5065408812039385), np.float64(5.811459118796061)), 'delivery_time_standardized': (np.float64(-0.061979503230456014), np.float64(0.06197950323045629)), 'delivery_time_standardized2': (np.float64(-0.06201051624377051), np.float64(0.06201051624377079))}


# Confidence Intervals with Small samples

In [12]:
# Take a sample from the data
sample = df.sample(20)
sample.describe()

,Toppings,Price,Delivery Time,# pizzas the customer ordered before,delivery_time_standardized,delivery_time_standardized2
count,20.00000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,4.15000,19.100000,25.230000,5.050000,0.067821,0.067854
std,0.74516,3.110255,3.232581,2.038446,1.298018,1.298668
min,3.00000,14.000000,19.900000,2.000000,-2.072400,-2.073437
25%,4.00000,17.000000,21.875000,3.750000,-1.279354,-1.279994
50%,4.00000,18.500000,26.250000,5.000000,0.477394,0.477633
75%,4.25000,22.000000,27.575000,6.000000,1.009437,1.009942
max,6.00000,25.000000,31.100000,9.000000,2.424874,2.426087


In [15]:
# With the function scipy.stats
st.t.interval(confidence = 0.95,
              df = len(sample) - 1,
              loc = sample.Price.mean(),
              scale = st.sem(sample.Price))

(np.float64(17.644355999411633), np.float64(20.55564400058837))

In [17]:
# Exercise: adapt the function to calculate the Confidence Interval
# when the sample size is less than 30, using chatGPT 

